# 01 Bond Pricing Engine

## Project context

This notebook builds Phase 0 of the Fixed Income and Balance Sheet Analytics Platform. It uses a synthetic bond book to demonstrate fixed income cash flow modeling, bond pricing, yield-to-maturity calculation, and basic portfolio market value reporting.

The bond data is synthetic. It is not real bank, insurer, or issuer portfolio data.

## Fixed income objective

- Create a small synthetic bond book.
- Generate coupon and principal cash flows.
- Price each bond using discounted cash flows.
- Calculate yield-to-maturity from the model price.
- Summarize portfolio market value by rating and sector.
- Save outputs for Phase 1 risk analytics.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR
from src.cashflows import build_cashflow_table
from src.pricing import price_bond, price_bond_book
from src.visualization import (
    plot_bond_prices_by_rating,
    plot_portfolio_market_value_by_sector,
    plot_cashflow_schedule,
    plot_price_vs_yield,
)

bond_book_dir = OUTPUTS_DIR / "bond_book"
pricing_dir = OUTPUTS_DIR / "pricing"
bond_book_dir.mkdir(parents=True, exist_ok=True)
pricing_dir.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

settlement_date = pd.Timestamp("2026-04-30")

## Synthetic bond book creation

In [ ]:
bond_book = pd.DataFrame(
    [
        ["BOND001", "Sovereign", "Government", 1_000_000, 0.045, 2, "2021-01-15", "2028-01-15", 0.050, "AA", 980_000],
        ["BOND002", "Bank", "Financials", 750_000, 0.055, 2, "2022-03-01", "2029-03-01", 0.061, "A", 735_000],
        ["BOND003", "Corporate", "Utilities", 600_000, 0.062, 2, "2020-06-30", "2030-06-30", 0.066, "A", 590_000],
        ["BOND004", "Corporate", "Property", 500_000, 0.070, 4, "2023-09-15", "2031-09-15", 0.078, "BBB", 480_000],
        ["BOND005", "Sovereign", "Government", 1_250_000, 0.038, 2, "2019-11-20", "2027-11-20", 0.044, "AA", 1_220_000],
        ["BOND006", "Corporate", "Consumer", 450_000, 0.065, 2, "2024-02-10", "2032-02-10", 0.071, "BBB", 430_000],
        ["BOND007", "Bank", "Financials", 900_000, 0.052, 2, "2021-07-01", "2031-07-01", 0.058, "A", 880_000],
        ["BOND008", "Corporate", "Telecom", 700_000, 0.060, 2, "2022-12-05", "2034-12-05", 0.069, "BBB", 670_000],
        ["BOND009", "Agency", "Government-Linked", 800_000, 0.047, 2, "2020-04-25", "2028-04-25", 0.052, "AA", 790_000],
        ["BOND010", "Corporate", "Infrastructure", 550_000, 0.075, 4, "2025-01-15", "2035-01-15", 0.083, "BBB", 520_000],
    ],
    columns=[
        "bond_id",
        "issuer_type",
        "sector",
        "face_value",
        "coupon_rate",
        "coupon_frequency",
        "issue_date",
        "maturity_date",
        "market_yield",
        "credit_rating",
        "book_value",
    ],
)
for col in ["issue_date", "maturity_date"]:
    bond_book[col] = pd.to_datetime(bond_book[col])
bond_book.to_csv(PROCESSED_DATA_DIR / "synthetic_bond_book.csv", index=False)
bond_book

## Cash flow generation

In [ ]:
cashflow_table = build_cashflow_table(bond_book, settlement_date=settlement_date)
cashflow_table.to_csv(bond_book_dir / "bond_cashflows.csv", index=False)
cashflow_table.head(), cashflow_table.shape

## Bond pricing

In [ ]:
pricing_results = price_bond_book(bond_book, settlement_date=settlement_date)
pricing_results.to_csv(pricing_dir / "bond_pricing_results.csv", index=False)
pricing_results[["bond_id", "credit_rating", "sector", "clean_price_per_100", "market_yield", "calculated_ytm", "market_value"]]

## Yield-to-maturity calculation

The calculated YTM is recovered from the model price. It should be close to the synthetic market yield used to price each bond.

In [ ]:
pricing_results["ytm_difference"] = pricing_results["calculated_ytm"] - pricing_results["market_yield"]
pricing_results[["bond_id", "market_yield", "calculated_ytm", "ytm_difference"]]

## Portfolio market value summary

In [ ]:
portfolio_summary = pd.DataFrame(
    [
        {
            "settlement_date": settlement_date,
            "bond_count": len(pricing_results),
            "total_face_value": pricing_results["face_value"].sum(),
            "total_book_value": pricing_results["book_value"].sum(),
            "total_market_value": pricing_results["market_value"].sum(),
            "market_to_book_ratio": pricing_results["market_value"].sum() / pricing_results["book_value"].sum(),
            "weighted_average_coupon": np.average(pricing_results["coupon_rate"], weights=pricing_results["face_value"]),
            "weighted_average_yield": np.average(pricing_results["market_yield"], weights=pricing_results["market_value"]),
        }
    ]
)
portfolio_summary.to_csv(pricing_dir / "portfolio_pricing_summary.csv", index=False)
portfolio_summary.T

## Pricing by rating and sector

In [ ]:
fig = plot_bond_prices_by_rating(pricing_results)
fig.savefig(FIGURES_DIR / "bond_prices_by_rating.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
fig = plot_portfolio_market_value_by_sector(pricing_results)
fig.savefig(FIGURES_DIR / "market_value_by_sector.png", dpi=150, bbox_inches="tight")
fig

## Cash flow schedule

In [ ]:
fig = plot_cashflow_schedule(cashflow_table)
fig.savefig(FIGURES_DIR / "cashflow_schedule.png", dpi=150, bbox_inches="tight")
fig

## Price versus yield example

In [ ]:
example_bond = bond_book.iloc[0]
yield_grid = np.linspace(0.01, 0.10, 60)
price_grid = [
    price_bond(
        face_value=example_bond["face_value"],
        coupon_rate=example_bond["coupon_rate"],
        market_yield=y,
        maturity_date=example_bond["maturity_date"],
        settlement_date=settlement_date,
        coupon_frequency=example_bond["coupon_frequency"],
    )
    for y in yield_grid
]
fig = plot_price_vs_yield(yield_grid, price_grid)
fig.savefig(FIGURES_DIR / "price_vs_yield.png", dpi=150, bbox_inches="tight")
fig

## Business interpretation

The synthetic portfolio shows how bond value depends on coupon structure, maturity, and market yield. Bonds with coupon rates below market yields tend to price below par, while higher coupon bonds can remain closer to par depending on maturity. Sector and rating summaries give a first view of portfolio concentration before adding duration, convexity, and stress testing.

## Limitations

- Synthetic data only.
- Simple Actual/365 timing.
- Flat market yield per bond rather than a real yield curve.
- No accrued interest, settlement calendar, duration, convexity, or stress testing yet.

## Next steps for Phase 1

- Add duration and convexity.
- Add rate shock scenarios.
- Summarize portfolio sensitivity by sector and rating.
- Prepare risk analytics outputs for final reports.